# Research 2: LoRA teacher and activation transfer
Upload the minimal bundle built with `python -m sp_lense.research2.package <output.zip>`. Verify its digest before execution. This recorded notebook pins the exact executed bundle.

In [ ]:
import torch, subprocess, sys
print(torch.cuda.get_device_name(0), torch.__version__)
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==5.15.1','peft==0.18.1'])


In [ ]:
from pathlib import Path
import zipfile, hashlib, json, os, sys, subprocess, time, shutil
from IPython.display import clear_output
archive = Path('/content/research2-pilot-ready.zip')
assert hashlib.sha256(archive.read_bytes()).hexdigest() == 'ad5e71905d71ef94f48ddc2001ac99d046f8ecfc4a6e274c6a3040916b5faa91'
root = Path('/content/research2')
root.mkdir(exist_ok=False)
with zipfile.ZipFile(archive) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(root)
for name, digest in json.loads((root/'PILOT_MANIFEST.json').read_text()).items():
    assert hashlib.sha256((root/name).read_bytes()).hexdigest() == digest, name
out = root/'work/run_01'
env = dict(os.environ, SP_LENSE_REPO=str(root), PYTHONPATH=str(root/'src'))
runner = "import json; from pathlib import Path; from sp_lense.research2.runtime import main; from sp_lense.research2.report import write; r=Path('/content/research2'); main(r,r/'work/run_01',json.loads((r/'study/02_lora_transfer/config.json').read_text())); write(r/'work/run_01')"
log = (root/'execution.log').open('w')
process = subprocess.Popen([sys.executable,'-u','-c',runner],cwd=root,env=env,stdout=log,stderr=subprocess.STDOUT)
started = time.monotonic()
try:
    while process.poll() is None:
        if time.monotonic()-started > 7200:
            process.kill()
            process.wait()
            raise TimeoutError('Hard two-hour wall-time cap reached')
        status = out/'STATUS.json'
        if status.exists():
            clear_output(wait=True)
            print('RESEARCH2_STATUS '+status.read_text(), flush=True)
        time.sleep(5)
    log.close()
    clear_output(wait=True)
    print('RESEARCH2_EXIT',process.returncode)
    if (out/'RESULT.md').exists(): print((out/'RESULT.md').read_text())
    elif (out/'FAILURE.json').exists(): print((out/'FAILURE.json').read_text())
    print((root/'execution.log').read_text()[-6000:])
finally:
    log.close()
    if out.exists():
        shutil.copy2(root/'execution.log',out/'execution.log')
        shutil.make_archive('/content/research2-results','zip',out)
        print('RESULT_ARCHIVE /content/research2-results.zip')
